# Production Export, Merging Protocols & Performance Tradeoffs

## QLoRA vs. BF16 LoRA

Deploying a QLoRA-adapted model requires an explicit understanding of quantization boundaries during weight export, alongside a clear assessment of performance and speed tradeoffs relative to standard 16-bit LoRA.

---

## The QLoRA Merge Imperative: Dequantize First

Attempting to merge 16-bit LoRA weights ($B \cdot A$) directly into a 4-bit base model ($W_{\text{NF4}}$) in-place is mathematically invalid.

**INCORRECT QLoRA MERGE (Corrupts Model):**

```
[ 4-bit NF4 Base W_0 ]  +  [ 16-bit BF16 Adapter (B * A) ]  ──► CATASTROPHIC DRIFT / GARBAGE OUTPUT
```

**CORRECT QLoRA MERGE PIPELINE:**

```
[ 4-bit NF4 Base W_0 ] ──► [ Dequantize to FP32 ] ──┐
                                                    ├──► [ W_merged (FP32) ] ──► [ Downcast to BF16 ]
[ 16-bit BF16 Adapter ] ──► [ Upcast to FP32 ] ────┘
```

---

### Production Export & Merge Protocol

**Step 1: Load Base Model in Full Unquantized Precision (FP16 or BF16)**

* Do not pass `load_in_4bit=True` or bitsandbytes quantization flags when loading the base model for merging
* Load raw 16-bit base weights $W_0$

**Step 2: Attach Trained LoRA Adapters**

* Load the saved `adapter_model.safetensors` onto the unquantized base model

**Step 3: Execute Precision-Safe FP32 Merge**

$$W_{\text{merged}} = W_{0\text{\_fp32}} + \left(\frac{\alpha}{r}\right) \left(B_{\text{fp32}} \cdot A_{\text{fp32}}\right)$$

**Step 4: Export Clean Checkpoint**

* Save $W_{\text{merged}}$ as a standalone unquantized safetensors checkpoint (e.g., in bfloat16 format)

**Step 5: Post-Merge Quantization (Optional)**

* If low-bit deployment is required for serving (e.g., AWQ, GPTQ, GGUF, or FP8 execution), perform post-training quantization on the newly merged checkpoint

---

## Tradeoff Analysis: Standard BF16 LoRA vs. QLoRA

While QLoRA unlocks massive VRAM savings, it introduces specific computational costs and minor precision degradation.

| Metric / Aspect | BF16 LoRA (16-bit Base + 16-bit Adapter) | QLoRA (4-bit NF4 Base + 16-bit Adapter) |
|---|---|---|
| **Base Model VRAM Footprint** | $2.0 \text{ bytes / parameter}$ ($\sim 14\text{ GB}$ for 7B) | $0.5 \text{ bytes / parameter}$ ($\sim 3.8\text{ GB}$ for 7B) |
| **Training Speed (Tokens/sec)** | Baseline (100%) — Direct Tensor Core GEMM | ~70% - 80% of BF16 Speed — Overhead from 4-bit → 16-bit register dequantization per layer |
| **Benchmark Accuracy** | Matches Full Fine-Tuning | ~99.0% - 99.5% of Full Fine-Tuning (Negligible loss for most instruction tasks) |
| **Hardware Requirements** | Requires higher-tier VRAM GPUs (A100/H100 for 70B models) | Runs on consumer/single-node GPUs (RTX 3090 / 4090 / L4) |
| **Dependency Overhead** | Native PyTorch / Hugging Face PEFT | Requires CUDA dequantization library (bitsandbytes or Unsloth Triton kernels) |

---

## Engineering Verdict

**Use BF16 LoRA if:**
* You have abundant multi-GPU hardware (e.g., A100/H100 clusters)
* High-throughput speed is paramount
* You are training on high-complexity math/coding reasoning tasks where sub-percent precision loss matters

**Use QLoRA if:**
* Training on a single GPU
* Fine-tuning large 70B+ models
* Performing routine instruction alignment, style transfer, and JSON schema formatting tasks